**Importante correr el código para para acceder a a las carpetas del drive**

In [1]:
from google.colab import drive
import os

# Attempt to unmount if already mounted or clean up directory if it exists
if os.path.exists('/content/drive'):
    if os.path.ismount('/content/drive'):
        print('Unmounting existing Google Drive...')
        !fusermount -uz /content/drive
        # Give a moment for the unmount to complete
        import time
        time.sleep(1)
    elif os.path.isdir('/content/drive') and os.listdir('/content/drive'):
        print('Cleaning up /content/drive directory...')
        # Use rmdir if empty, or rm -rf if not empty (be careful with rm -rf)
        # For this specific error, it implies files, so rm -rf is safer here.
        !rm -rf /content/drive/*

drive.mount('/content/drive', force_remount=True)

Unmounting existing Google Drive...
Mounted at /content/drive


**Antes de realizar el filtrado se crea la columna de PL**

# **FILTRADO, DEPURACION Y DIVISION**
Outlayer usando qIQR: https://procogia.com/interquartile-range-method-for-reliable-data-analysis/

**Filtrado**

Se elimina datos en blanco, filtramos columnas inecesarias, y tambien seleccionamos Uplink o Donwlink

**División de dataset**

Dividimos en tres partes: Entrenamiento, validación testeo

**Esto hace de todo el dataset NO POR LINEAS**




Before filtering, the PL column is created.
#**FILTERING, CLEANING, AND SPLITTING**
Outlayer using qIQR: https://procogia.com/interquartile-range-method-for-reliable-data-analysis/

Filtering

Blank data is removed, unnecessary columns are filtered, and we also select Uplink or Downlink

Splitting the dataset

We divide it into three parts: training, validation, and testing.

This makes the entire dataset NOT by lines.

# **UPLINK**

In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
import sys

# ==============================================================================
# 0. CONFIGURACIÓN
# ==============================================================================
RUTA_ARCHIVO = '/content/drive/MyDrive/TESIS 100%/dataset_separado_ordenado.xlsx'
CARPETA_SALIDA_RAIZ = '/content/drive/MyDrive/TESIS 100%/Datos_Entrenamiento_Finales/'

if not os.path.exists(CARPETA_SALIDA_RAIZ): os.makedirs(CARPETA_SALIDA_RAIZ)

print("INICIANDO LIMPIEZA Y DIVISIÓN (UPLINK)...")

# ==============================================================================
# 1. CARGA OBLIGATORIA
# ==============================================================================
try:
    df_raw = pd.read_excel(RUTA_ARCHIVO)
    print(f"\n✅ Archivo cargado: {len(df_raw)} filas originales.")
except Exception as e:
    print(f"\n❌ ERROR FATAL: No se pudo cargar el archivo: {e}")
    sys.exit("PROCESO DETENIDO")

# Limpieza inicial de nombres de columnas
df_raw.columns = [c.replace('"', '').replace("'", "").strip() for c in df_raw.columns]

# Corrección de fecha
try:
    df_raw['Timestamp'] = pd.to_datetime(df_raw['Fecha'].astype(str).str.split(' ').str[0] + ' ' + df_raw['Hora'].astype(str), errors='coerce')
except:
    pass

df_raw.rename(columns={'Distancia km': 'Distance', 'Tipo de terreno': 'Terreno_Desc', 'Terreno': 'Terreno'}, inplace=True)

# Conversión Km -> Metros
if df_raw['Distance'].mean() < 100:
    df_raw['Distance'] = df_raw['Distance'] * 1000

# ==============================================================================
# FUNCIÓN PRINCIPAL DE PROCESAMIENTO
# ==============================================================================
def procesar_flujo_uplink(df_base):
    flujo = 'UpLink'
    print(f"\n" + "="*50)
    print(f"PROCESANDO FLUJO: {flujo.upper()}")
    print("="*50)

    CARPETA_SALIDA = os.path.join(CARPETA_SALIDA_RAIZ, flujo)
    if not os.path.exists(CARPETA_SALIDA): os.makedirs(CARPETA_SALIDA)

    df = df_base.copy()

    # Definir columnas
    sf_col = 'SpreedFactor_UpLink' if 'SpreedFactor_UpLink' in df.columns else 'SpreedFactor'
    pow_col = 'Power_UpLink'
    rssi_col = 'RSSI_UpLink'
    snr_col = 'SNR_UpLink'
    target_col = 'Path_Loss_UpLink'

    if sf_col not in df.columns and 'SpreedFactor' in df.columns:
        df.rename(columns={'SpreedFactor': sf_col}, inplace=True)

    cols_vitales = ['Distance', sf_col, 'Terreno', pow_col, rssi_col, snr_col]

    # --- DETECTOR DE COLUMNAS FALTANTES ---
    faltantes = [c for c in cols_vitales if c not in df.columns]
    if faltantes:
        print(f" ¡Atención! Faltan columnas: {faltantes}")
        return

    # --- FILTROS BÁSICOS ---
    filas_antes_nulos = len(df)
    df = df.dropna(subset=cols_vitales).copy()
    print(f"🧹 Filtro Nulos en columnas vitales: {filas_antes_nulos} -> {len(df)} (Eliminados: {filas_antes_nulos - len(df)})")

    conteo_antes_dist = len(df)
    df = df[(df['Distance'] >= 200) & (df['Distance'] <= 6000)].copy()
    print(f"📏 Filtro Distancia (200m - 6000m): {conteo_antes_dist} -> {len(df)} (Perdidos: {conteo_antes_dist - len(df)})")

    if len(df) == 0:
        print(f"⚠️ No quedaron datos.")
        return

    # --- CÁLCULO PATH LOSS ---
    def calc_pl(row):
        snr = row[snr_col]
        try: fact = 10 * np.log10(1 + 10**(snr/10))
        except: fact = 0
        return row[pow_col] + 7.3 - row[rssi_col] + snr - fact

    df[target_col] = df.apply(calc_pl, axis=1)

    # --- OUTLIERS (IQR) ---
    df['Dist_Bin'] = (np.round(df['Distance'] / 100) * 100).astype(int)
    df[sf_col] = df[sf_col].astype(int)
    df[pow_col] = df[pow_col].astype(int)

    print(f"\n🔍 Eliminando Outliers ({flujo})...")
    grupos = df.groupby([sf_col, pow_col, 'Dist_Bin'])
    df_list = []

    for _, grupo in grupos:
        if len(grupo) < 3:
            df_list.append(grupo)
            continue
        Q1 = grupo[rssi_col].quantile(0.25)
        Q3 = grupo[rssi_col].quantile(0.75)
        IQR = Q3 - Q1
        mask = (grupo[rssi_col] >= Q1 - 1.0*IQR) & (grupo[rssi_col] <= Q3 + 1.0*IQR)
        df_list.append(grupo[mask])

    df_clean = pd.concat(df_list).drop(columns=['Dist_Bin'])
    print(f"✅ Datos tras limpieza IQR: {len(df_clean)} filas (Eliminados: {len(df) - len(df_clean)}).")

    # --- SPLIT INTELIGENTE ---
    print(f"\nDividiendo Train/Val/Test ({flujo})...")
    df_clean['Stratify_Key'] = "SF" + df_clean[sf_col].astype(str) + "_P" + df_clean[pow_col].astype(str)
    counts = df_clean['Stratify_Key'].value_counts()

    keys_validas = counts[counts >= 5].index
    df_main = df_clean[df_clean['Stratify_Key'].isin(keys_validas)].copy()
    df_orphans = df_clean[~df_clean['Stratify_Key'].isin(keys_validas)].copy()

    if len(df_orphans) > 0:
        print(f" ℹ️ Se encontraron {len(df_orphans)} datos 'huérfanos' (a Train).")

    # Split 70/15/15
    train_A, temp = train_test_split(df_main, test_size=0.3, stratify=df_main['Stratify_Key'], random_state=42)
    val, test = train_test_split(temp, test_size=0.5, stratify=temp['Stratify_Key'], random_state=42)

    train = pd.concat([train_A, df_orphans])

    # Etiquetas para gráficas
    train['Dataset'] = 'Train'
    val['Dataset'] = 'Validation'
    test['Dataset'] = 'Test'
    df_all = pd.concat([train, val, test])

    # --- GUARDAR CSVs ---
    cols_a_guardar = ['Timestamp', 'Distance', 'Gateway_Altitude', 'EndDevice_Altitude',
                      sf_col, 'Terreno', pow_col, rssi_col, snr_col, target_col]
    cols_finales = [c for c in cols_a_guardar if c in train.columns]

    # 1. CSV COMPLETO LIMPIO
    df_clean[cols_finales].to_csv(os.path.join(CARPETA_SALIDA, f'Full_{flujo}_Cleaned.csv'), index=False)

    # 2. CSVs DIVIDIDOS
    train[cols_finales].to_csv(os.path.join(CARPETA_SALIDA, f'Train_{flujo}.csv'), index=False)
    val[cols_finales].to_csv(os.path.join(CARPETA_SALIDA, f'Val_{flujo}.csv'), index=False)
    test[cols_finales].to_csv(os.path.join(CARPETA_SALIDA, f'Test_{flujo}.csv'), index=False)

    # --- REPORTAR (MISMO FORMATO QUE TU SCRIPT ANTERIOR) ---
    total_datos = len(df_all)
    pct_train = (len(train) / total_datos) * 100
    pct_val = (len(val) / total_datos) * 100
    pct_test = (len(test) / total_datos) * 100

    print(f"\n✅ ¡PROCESO TERMINADO PARA {flujo.upper()}!")
    print(f" 📊 TOTAL DATOS LIMPIOS: {total_datos} (100.00%)")
    print(f" 📂 Train: {len(train):>6} datos ({pct_train:>6.2f}%)")
    print(f" 📂 Val:   {len(val):>6} datos ({pct_val:>6.2f}%)")
    print(f" 📂 Test:  {len(test):>6} datos ({pct_test:>6.2f}%)")

    # ==========================================================================
    # GENERACIÓN DE DASHBOARD DE VERIFICACIÓN
    # ==========================================================================
    print(f"\n📊 Generando Dashboard de Verificación de Muestreo...")
    plt.rcParams['font.family'] = 'serif'

    fig = plt.figure(figsize=(18, 12))
    gs = fig.add_gridspec(2, 2, hspace=0.3, wspace=0.2)

    # 1. CONTEO POR ARCHIVO
    ax1 = fig.add_subplot(gs[0, 0])
    counts = [len(df_clean), len(train), len(val), len(test)]
    labels = ['Completo', 'Train (70%)', 'Validation (15%)', 'Test (15%)']
    colors = ['gray', '#2ecc71', '#3498db', '#e74c3c']

    bars = ax1.bar(labels, counts, color=colors, edgecolor='black', alpha=0.8)
    ax1.bar_label(bars, fmt='%d', fontsize=11, fontweight='bold')
    ax1.set_title('Cantidad de Datos por Archivo', fontsize=14, fontweight='bold')
    ax1.set_ylabel('Número de Muestras', fontsize=12)
    ax1.grid(axis='y', linestyle='--', alpha=0.4)

    # 2. DISTRIBUCIÓN POR SF
    ax2 = fig.add_subplot(gs[0, 1])
    order_sf = sorted(df_all[sf_col].unique())
    sns.countplot(data=df_all, x=sf_col, hue='Dataset', hue_order=['Train', 'Validation', 'Test'],
                  palette={'Train': '#2ecc71', 'Validation': '#3498db', 'Test': '#e74c3c'},
                  ax=ax2, edgecolor='black', order=order_sf)
    ax2.set_title('Verificación de Estratificación por SF', fontsize=14, fontweight='bold')
    ax2.set_xlabel('Spreading Factor', fontsize=12)
    ax2.legend(title='Dataset')
    ax2.grid(axis='y', linestyle='--', alpha=0.4)

    # 3. DISTRIBUCIÓN POR POTENCIA
    ax3 = fig.add_subplot(gs[1, 0])
    order_pwr = sorted(df_all[pow_col].unique())
    sns.countplot(data=df_all, x=pow_col, hue='Dataset', hue_order=['Train', 'Validation', 'Test'],
                  palette={'Train': '#2ecc71', 'Validation': '#3498db', 'Test': '#e74c3c'},
                  ax=ax3, edgecolor='black', order=order_pwr)
    ax3.set_title('Verificación de Estratificación por Potencia', fontsize=14, fontweight='bold')
    ax3.set_xlabel('Potencia de Transmisión (dBm)', fontsize=12)
    ax3.legend(title='Dataset')
    ax3.grid(axis='y', linestyle='--', alpha=0.4)

    # 4. INTEGRIDAD COMPLETA (SF + POTENCIA)
    ax4 = fig.add_subplot(gs[1, 1])
    cross_tab = pd.crosstab(df_all['Stratify_Key'], df_all['Dataset'])
    cross_tab = cross_tab[['Train', 'Validation', 'Test']]

    cross_tab.plot(kind='bar', stacked=True, color=['#2ecc71', '#3498db', '#e74c3c'],
                   ax=ax4, width=0.8, edgecolor='black')
    ax4.set_title('Detalle Completo (SF + Potencia)', fontsize=14, fontweight='bold')
    ax4.set_xlabel('Configuración', fontsize=12)
    ax4.legend(title='Dataset')
    plt.xticks(rotation=45, ha='right', fontsize=9)
    ax4.grid(axis='y', linestyle='--', alpha=0.4)

    # Guardar
    fig.suptitle(f'REPORTE DE INTEGRIDAD Y MUESTREO - {flujo}', fontsize=18, fontweight='bold', y=0.98)
    ruta_grafica = os.path.join(CARPETA_SALIDA, f'Dashboard_Validacion_Muestreo_{flujo}.png')
    plt.savefig(ruta_grafica, dpi=300, bbox_inches='tight')
    plt.close()

    print(f"✅ Dashboard guardado en: {ruta_grafica}")

# ==============================================================================
# EJECUCIÓN
# ==============================================================================
procesar_flujo_uplink(df_raw)

# **DOWNLINK**

In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
import sys

# ==============================================================================
# 0. CONFIGURACIÓN
# ==============================================================================
RUTA_ARCHIVO = '/content/drive/MyDrive/TESIS 100%/dataset_separado_ordenado.xlsx'
CARPETA_SALIDA_RAIZ = '/content/drive/MyDrive/TESIS 100%/Datos_Entrenamiento_Finales/'

if not os.path.exists(CARPETA_SALIDA_RAIZ): os.makedirs(CARPETA_SALIDA_RAIZ)

print("INICIANDO PROCESO DEFINITIVO DE LIMPIEZA Y DIVISIÓN (DOWNLINK)...")

# ==============================================================================
# 1. CARGA OBLIGATORIA
# ==============================================================================
try:
    df_raw = pd.read_excel(RUTA_ARCHIVO)
    print(f"\n✅ Archivo cargado: {len(df_raw)} filas originales.")
except Exception as e:
    print(f"\n❌ ERROR FATAL: No se pudo cargar el archivo: {e}")
    sys.exit("PROCESO DETENIDO")

# Limpieza inicial de nombres de columnas
df_raw.columns = [c.replace('"', '').replace("'", "").strip() for c in df_raw.columns]

# Corrección de fecha
try:
    df_raw['Timestamp'] = pd.to_datetime(df_raw['Fecha'].astype(str).str.split(' ').str[0] + ' ' + df_raw['Hora'].astype(str), errors='coerce')
except:
    pass

df_raw.rename(columns={'Distancia km': 'Distance', 'Tipo de terreno': 'Terreno_Desc', 'Terreno': 'Terreno'}, inplace=True)

# Conversión Km -> Metros
if df_raw['Distance'].mean() < 100:
    df_raw['Distance'] = df_raw['Distance'] * 1000

# ==============================================================================
# FUNCIÓN PRINCIPAL DE PROCESAMIENTO (DOWNLINK)
# ==============================================================================
def procesar_flujo_downlink(df_base):
    flujo = 'DownLink'
    print(f"\n" + "="*50)
    print(f"PROCESANDO FLUJO: {flujo.upper()}")
    print("="*50)

    CARPETA_SALIDA = os.path.join(CARPETA_SALIDA_RAIZ, flujo)
    if not os.path.exists(CARPETA_SALIDA): os.makedirs(CARPETA_SALIDA)

    df = df_base.copy()

    # --- CONFIGURACIÓN DE COLUMNAS ---
    # 1. Spreading Factor (Si no hay DL, usamos UL o Genérico)
    sf_col = 'SpreedFactor_DownLink'
    if sf_col not in df.columns:
        if 'SpreedFactor_UpLink' in df.columns:
            sf_col = 'SpreedFactor_UpLink' # Reutilizamos UL
        elif 'SpreedFactor' in df.columns:
            sf_col = 'SpreedFactor'

    # 2. Potencia (26 dBm)
    pow_col = 'Power_DownLink'
    print("ℹConfigurando Potencia DownLink fija en 26 dBm.")
    df[pow_col] = 26 # Valor constante para todo el dataset

    # 3. RSSI y SNR (Deben ser las de DownLink)
    rssi_col = 'RSSI_DownLink'
    snr_col = 'SNR_DownLink'
    target_col = 'Path_Loss_DownLink'

    cols_vitales = ['Distance', sf_col, 'Terreno', pow_col, rssi_col, snr_col]

    # --- DETECTOR DE COLUMNAS FALTANTES ---
    faltantes = [c for c in cols_vitales if c not in df.columns]
    if faltantes:
        print(f"⚠️ ¡Atención! Faltan columnas: {faltantes}")
        return

    # --- FILTROS BÁSICOS ---
    filas_antes_nulos = len(df)
    df = df.dropna(subset=cols_vitales).copy()
    print(f"🧹 Filtro Nulos en columnas vitales: {filas_antes_nulos} -> {len(df)} (Eliminados: {filas_antes_nulos - len(df)})")

    conteo_antes_dist = len(df)
    df = df[(df['Distance'] >= 200) & (df['Distance'] <= 6000)].copy()
    print(f"📏 Filtro Distancia (200m - 6000m): {conteo_antes_dist} -> {len(df)} (Perdidos: {conteo_antes_dist - len(df)})")

    if len(df) == 0:
        print(f"⚠️ No quedaron datos.")
        return

    # --- CÁLCULO PATH LOSS ---
    def calc_pl(row):
        snr = row[snr_col]
        try: fact = 10 * np.log10(1 + 10**(snr/10))
        except: fact = 0
        return row[pow_col] + 7.3 - row[rssi_col] + snr - fact

    df[target_col] = df.apply(calc_pl, axis=1)

    # --- OUTLIERS (IQR) ---
    df['Dist_Bin'] = (np.round(df['Distance'] / 100) * 100).astype(int)
    df[sf_col] = df[sf_col].astype(int)
    # Nota: No agrupamos por Power porque es constante (26), solo por SF y Distancia

    print(f"\n🔍 Eliminando Outliers ({flujo})...")
    grupos = df.groupby([sf_col, 'Dist_Bin'])
    df_list = []

    for _, grupo in grupos:
        if len(grupo) < 3:
            df_list.append(grupo)
            continue
        Q1 = grupo[rssi_col].quantile(0.25)
        Q3 = grupo[rssi_col].quantile(0.75)
        IQR = Q3 - Q1
        mask = (grupo[rssi_col] >= Q1 - 1.0*IQR) & (grupo[rssi_col] <= Q3 + 1.0*IQR)
        df_list.append(grupo[mask])

    df_clean = pd.concat(df_list).drop(columns=['Dist_Bin'])
    print(f"✅ Datos tras limpieza IQR: {len(df_clean)} filas (Eliminados: {len(df) - len(df_clean)}).")

    # --- SPLIT---
    print(f"\nDividiendo Train/Val/Test...")
    # Stratify solo por SF (Power es único)
    df_clean['Stratify_Key'] = "SF" + df_clean[sf_col].astype(str)
    counts = df_clean['Stratify_Key'].value_counts()

    keys_validas = counts[counts >= 5].index
    df_main = df_clean[df_clean['Stratify_Key'].isin(keys_validas)].copy()
    df_orphans = df_clean[~df_clean['Stratify_Key'].isin(keys_validas)].copy()

    if len(df_orphans) > 0:
        print(f" ℹ️ Se encontraron {len(df_orphans)} datos 'huérfanos' (a Train).")

    # Split 70/15/15
    train_A, temp = train_test_split(df_main, test_size=0.3, stratify=df_main['Stratify_Key'], random_state=42)
    val, test = train_test_split(temp, test_size=0.5, stratify=temp['Stratify_Key'], random_state=42)

    train = pd.concat([train_A, df_orphans])

    # Etiquetas para gráficas
    train['Dataset'] = 'Train'
    val['Dataset'] = 'Validation'
    test['Dataset'] = 'Test'
    df_all = pd.concat([train, val, test])

    # --- GUARDAR CSVs ---
    cols_a_guardar = ['Timestamp', 'Distance', 'Gateway_Altitude', 'EndDevice_Altitude',
                      sf_col, 'Terreno', pow_col, rssi_col, snr_col, target_col]
    cols_finales = [c for c in cols_a_guardar if c in train.columns]

    # 1. CSV COMPLETO LIMPIO
    df_clean[cols_finales].to_csv(os.path.join(CARPETA_SALIDA, f'Full_{flujo}_Cleaned.csv'), index=False)

    # 2. CSVs DIVIDIDOS
    train[cols_finales].to_csv(os.path.join(CARPETA_SALIDA, f'Train_{flujo}.csv'), index=False)
    val[cols_finales].to_csv(os.path.join(CARPETA_SALIDA, f'Val_{flujo}.csv'), index=False)
    test[cols_finales].to_csv(os.path.join(CARPETA_SALIDA, f'Test_{flujo}.csv'), index=False)

    # --- REPORTAR ---
    total_datos = len(df_all)
    pct_train = (len(train) / total_datos) * 100
    pct_val = (len(val) / total_datos) * 100
    pct_test = (len(test) / total_datos) * 100

    print(f"\n✅ ¡PROCESO TERMINADO PARA {flujo.upper()}!")
    print(f" 📊 TOTAL DATOS LIMPIOS: {total_datos} (100.00%)")
    print(f" 📂 Train: {len(train):>6} datos ({pct_train:>6.2f}%)")
    print(f" 📂 Val:   {len(val):>6} datos ({pct_val:>6.2f}%)")
    print(f" 📂 Test:  {len(test):>6} datos ({pct_test:>6.2f}%)")

    # ==========================================================================
    # GENERACIÓN DE DASHBOARD DE VERIFICACIÓN
    # ==========================================================================
    print(f"\n📊 Generando Dashboard de Verificación de Muestreo...")
    plt.rcParams['font.family'] = 'serif'

    fig = plt.figure(figsize=(18, 12))
    gs = fig.add_gridspec(2, 2, hspace=0.3, wspace=0.2)

    # 1. CONTEO POR ARCHIVO
    ax1 = fig.add_subplot(gs[0, 0])
    counts = [len(df_clean), len(train), len(val), len(test)]
    labels = ['Completo', 'Train (70%)', 'Validation (15%)', 'Test (15%)']
    colors = ['gray', '#2ecc71', '#3498db', '#e74c3c']

    bars = ax1.bar(labels, counts, color=colors, edgecolor='black', alpha=0.8)
    ax1.bar_label(bars, fmt='%d', fontsize=11, fontweight='bold')
    ax1.set_title('Cantidad de Datos por Archivo', fontsize=14, fontweight='bold')
    ax1.set_ylabel('Número de Muestras', fontsize=12)
    ax1.grid(axis='y', linestyle='--', alpha=0.4)

    # 2. DISTRIBUCIÓN POR SF
    ax2 = fig.add_subplot(gs[0, 1])
    order_sf = sorted(df_all[sf_col].unique())
    sns.countplot(data=df_all, x=sf_col, hue='Dataset', hue_order=['Train', 'Validation', 'Test'],
                  palette={'Train': '#2ecc71', 'Validation': '#3498db', 'Test': '#e74c3c'},
                  ax=ax2, edgecolor='black', order=order_sf)
    ax2.set_title('Verificación de Estratificación por SF', fontsize=14, fontweight='bold')
    ax2.set_xlabel('Spreading Factor', fontsize=12)
    ax2.legend(title='Dataset')
    ax2.grid(axis='y', linestyle='--', alpha=0.4)

    # 3. DISTRIBUCIÓN POR POTENCIA (MONOCROMÁTICA - SÓLO HAY 1 VALOR)
    ax3 = fig.add_subplot(gs[1, 0])
    # Como solo hay 26 dBm, el countplot mostrará una sola barra agrupada
    sns.countplot(data=df_all, x=pow_col, hue='Dataset', hue_order=['Train', 'Validation', 'Test'],
                  palette={'Train': '#2ecc71', 'Validation': '#3498db', 'Test': '#e74c3c'},
                  ax=ax3, edgecolor='black')
    ax3.set_title('Verificación de Potencia (Constante)', fontsize=14, fontweight='bold')
    ax3.set_xlabel('Potencia de Transmisión (dBm)', fontsize=12)
    ax3.legend(title='Dataset')
    ax3.grid(axis='y', linestyle='--', alpha=0.4)

    # 4. INTEGRIDAD COMPLETA
    ax4 = fig.add_subplot(gs[1, 1])
    cross_tab = pd.crosstab(df_all['Stratify_Key'], df_all['Dataset'])
    cross_tab = cross_tab[['Train', 'Validation', 'Test']]

    cross_tab.plot(kind='bar', stacked=True, color=['#2ecc71', '#3498db', '#e74c3c'],
                   ax=ax4, width=0.8, edgecolor='black')
    ax4.set_title('Detalle Completo (Solo SF varía)', fontsize=14, fontweight='bold')
    ax4.set_xlabel('Configuración', fontsize=12)
    ax4.legend(title='Dataset')
    plt.xticks(rotation=45, ha='right', fontsize=9)
    ax4.grid(axis='y', linestyle='--', alpha=0.4)

    # Guardar
    fig.suptitle(f'REPORTE DE INTEGRIDAD Y MUESTREO - {flujo}', fontsize=18, fontweight='bold', y=0.98)
    ruta_grafica = os.path.join(CARPETA_SALIDA, f'Dashboard_Validacion_Muestreo_{flujo}.png')
    plt.savefig(ruta_grafica, dpi=300, bbox_inches='tight')
    plt.close()

    print(f"✅ Dashboard guardado en: {ruta_grafica}")

# ==============================================================================
# EJECUCIÓN
# ==============================================================================
procesar_flujo_downlink(df_raw)